In [10]:
import torch
from torch import nn
import math
import copy
import numpy as np

In [6]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [13]:
def attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, v)

In [14]:
class MultiheadAttention(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        assert demb % h == 0
        self.d_k = demb // h
        self.h = h
        self.demb = demb
        self.linears = clones(nn.Linear(demb, demb), 4)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1) #adds a dimension to mask at second position for head axis
        nbatches = query.size(0)

        query, key, value = [linear(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) for linear, x in zip(self.linears, (query, key, value))]
        """ 
            this results in query, key, value of shape (batch_size, num_heads, seq_len, d_k) 
            taking dv=dk=demb//h
        """

        """
            contiguous function maintains continuty in memory space even after transpose, or else it might show an error with view function is used to reshape the tensor to the desired shape directly from memory
        """
        x = attention(query, key, value, mask=mask, dropout=self.dropout) #type:ignore
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.demb)
        return self.linears[-1](x)

In [28]:
np.random.seed(42)
x = np.random.randn(3, 4)
mha = MultiheadAttention(demb=4, h=2)
X = torch.tensor(x, dtype=torch.float32)
print(X)
mask = np.tril(np.ones((3, 4)))
mask = torch.tensor(mask, dtype=torch.bool) #lower triangular matrix of ones for masking future tokens in decoder
mha.forward(X, X, X, mask) #this will be the output to the attention layer seq_len*demb 

tensor([[ 0.4967, -0.1383,  0.6477,  1.5230],
        [-0.2342, -0.2341,  1.5792,  0.7674],
        [-0.4695,  0.5426, -0.4634, -0.4657]])


RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1

In [ ]:
import torch.nn.functional as F

"""
    SwiGLU(x, W, V) = [(xW + b) ⊙ σ(xW + b)] ⊙ (xV + c)
    FFN_SwiGLU(x) = SwiGLU(x, W, V) · W_out + b_out

    W, V : demb x dff
    W_out : dff x demb
    b,c : dff
    b_out : demb
    ⊙ - element-wise multiplication (Hadamard product)
"""

class SwiGLU(nn.Module):
    def __init__(self, demb):
        super().__init__()
        dff = demb * 8/3
        self.w_up = nn.Linear(demb, dff)
        self.w_gate = nn.Linear(demb, dff)
        self.w_down = nn.Linear(dff, demb)

        # often kaiming works best in cases of swish or silu, though we are using sigmoid so we have to experiment to choose betweeen kaiming and xavier initialization

        nn.init.kaiming_uniform_(self.w_gate.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_up.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_down.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.w_down(F.silu(self.w_up(x)) * self.w_gate(x))

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        self.attn = MultiheadAttention(demb, h, dropout)
        """
            the older tranformers used ff networks with linear layers and ReLU activation, but newer ones use SwiGLU which is a gated activation function that can capture more complex relationships in the data
        """
        self.dropout = nn.Dropout(p=dropout)
        self.norm = nn.RMSNorm(demb)

    def forward(self, x, mask=None):
        attn_output = self.attn.forward(x, x, x, mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm(x)

        ff = SwiGLU(x.size(-1))
        ff_output = ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm(x)

        return x

In [23]:
class Encoder(nn.Module):
    def __init__(self, demb, h, N, dropout=0.1):
        super().__init__()
        self.layers = clones(EncoderBlock(demb, h, dropout), N)
        self.norm = nn.RMSNorm(demb)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer.forward(x, mask=mask)
        return self.norm(x)